# Preparar a base para o dashboard


## Objetivo da etapa:

Criar um dataset limpo e focado em negócio que será consumido pelo Power BI ou Tableau.

Essa etapa envolve:
1. selecionar variáveis
2. remover colunas técnicas
3. salvar dataset final
4. preparar para visualização

### Importe e carregamento da base

In [1]:
import pandas as pd
import joblib

In [2]:
df_raw = pd.read_csv("/home/tatiane/repositorios/data-projects/03_machine_learning/etl_customer_value_classfication/raw/data/BankChurners.csv")
df_ml = pd.read_csv("/home/tatiane/repositorios/data-projects/03_machine_learning/etl_customer_value_classfication/raw/processed/customer_value_dataset.csv")

In [3]:
cols_to_remove = [
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2"
]

df_raw = df_raw.drop(columns=cols_to_remove)

In [4]:
print(df_raw.columns.tolist())

['CLIENTNUM', 'Attrition_Flag', 'Customer_Age', 'Gender', 'Dependent_count', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio']


In [5]:
print(df_ml.columns.tolist())

['CLIENTNUM', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Credit_Limit', 'Months_on_book', 'Months_Inactive_12_mon', 'customer_value_score', 'customer_value_class']


- Merge entre as bases

In [6]:
df_dashboard = df_raw.merge(
    df_ml[["CLIENTNUM", "customer_value_score", "customer_value_class"]],
    on="CLIENTNUM",
    how="left"
)

/tmp/ipykernel_142957/354754989.py:1: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  df_dashboard = df_raw.merge(


In [7]:
print(df_dashboard.columns.tolist())

['CLIENTNUM', 'Attrition_Flag', 'Customer_Age', 'Gender', 'Dependent_count', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'customer_value_score', 'customer_value_class']


### Selecionar variáveis úteis para BI

In [8]:
cols_dashboard = [
    "CLIENTNUM",

    # Segmentação (do ML)
    "customer_value_class",
    "customer_value_score",

    # Comportamento financeiro
    "Total_Trans_Amt",
    "Total_Trans_Ct",
    "Credit_Limit",
    "Avg_Utilization_Ratio",

    # Engajamento
    "Months_Inactive_12_mon",
    "Contacts_Count_12_mon",
    "Months_on_book",

    # Perfil do cliente
    "Customer_Age",
    "Gender",
    "Education_Level",
    "Income_Category",
    "Marital_Status",
    "Card_Category",

    # Risco
    "Attrition_Flag"
]

df_dashboard = df_dashboard[cols_dashboard]

In [9]:
print(df_dashboard.columns.tolist())

['CLIENTNUM', 'customer_value_class', 'customer_value_score', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Credit_Limit', 'Avg_Utilization_Ratio', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Months_on_book', 'Customer_Age', 'Gender', 'Education_Level', 'Income_Category', 'Marital_Status', 'Card_Category', 'Attrition_Flag']


### Padronização

In [10]:
rename_columns = {

    # identificador
    "CLIENTNUM": "customer_id",

    # segmentação (ML)
    "customer_value_class": "customer_value_segment",
    "customer_value_score": "customer_value_score",

    # comportamento financeiro
    "Total_Trans_Amt": "total_transaction_amount",
    "Total_Trans_Ct": "total_transaction_count",
    "Credit_Limit": "credit_limit",
    "Avg_Utilization_Ratio": "avg_credit_utilization",

    # engajamento
    "Months_Inactive_12_mon": "months_inactive_last_year",
    "Contacts_Count_12_mon": "contacts_last_year",
    "Months_on_book": "months_as_customer",

    # perfil
    "Customer_Age": "customer_age",
    "Gender": "gender",
    "Education_Level": "education_level",
    "Income_Category": "income_category",
    "Marital_Status": "marital_status",
    "Card_Category": "card_category",

    # churn
    "Attrition_Flag": "attrition_flag"
}

df_dashboard = df_dashboard.rename(columns=rename_columns)

### Análise básica

In [11]:
df_dashboard.head()

,customer_id,customer_value_segment,customer_value_score,total_transaction_amount,total_transaction_count,credit_limit,avg_credit_utilization,months_inactive_last_year,contacts_last_year,months_as_customer,customer_age,gender,education_level,income_category,marital_status,card_category,attrition_flag
0,768805383,NaN,NaN,1144,42,12691.0,0.061,1,3,39,45,M,High School,$60K - $80K,Married,Blue,Existing Customer
1,818770008,NaN,NaN,1291,33,8256.0,0.105,1,2,44,49,F,Graduate,Less than $40K,Single,Blue,Existing Customer
2,713982108,NaN,NaN,1887,20,3418.0,0.000,1,0,36,51,M,Graduate,$80K - $120K,Married,Blue,Existing Customer
3,769911858,NaN,NaN,1171,20,3313.0,0.760,4,1,34,40,F,High School,Less than $40K,Unknown,Blue,Existing Customer
4,709106358,NaN,NaN,816,28,4716.0,0.000,1,0,21,40,M,Uneducated,$60K - $80K,Married,Blue,Existing Customer


In [12]:
df_dashboard.describe()

,customer_id,customer_value_score,total_transaction_amount,total_transaction_count,credit_limit,avg_credit_utilization,months_inactive_last_year,contacts_last_year,months_as_customer,customer_age
count,1.012700e+04,0.0,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000
mean,7.391776e+08,NaN,4404.086304,64.858695,8631.953698,0.274894,2.341167,2.455317,35.928409,46.325960
std,3.690378e+07,NaN,3397.129254,23.472570,9088.776650,0.275691,1.010622,1.106225,7.986416,8.016814
min,7.080821e+08,NaN,510.000000,10.000000,1438.300000,0.000000,0.000000,0.000000,13.000000,26.000000
25%,7.130368e+08,NaN,2155.500000,45.000000,2555.000000,0.023000,2.000000,2.000000,31.000000,41.000000
50%,7.179264e+08,NaN,3899.000000,67.000000,4549.000000,0.176000,2.000000,2.000000,36.000000,46.000000
75%,7.731435e+08,NaN,4741.000000,81.000000,11067.500000,0.503000,3.000000,3.000000,40.000000,52.000000
max,8.283431e+08,NaN,18484.000000,139.000000,34516.000000,0.999000,6.000000,6.000000,56.000000,73.000000


In [13]:
df_dashboard.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer_id                10127 non-null  int64  
 1   customer_value_segment     0 non-null      object 
 2   customer_value_score       0 non-null      float64
 3   total_transaction_amount   10127 non-null  int64  
 4   total_transaction_count    10127 non-null  int64  
 5   credit_limit               10127 non-null  float64
 6   avg_credit_utilization     10127 non-null  float64
 7   months_inactive_last_year  10127 non-null  int64  
 8   contacts_last_year         10127 non-null  int64  
 9   months_as_customer         10127 non-null  int64  
 10  customer_age               10127 non-null  int64  
 11  gender                     10127 non-null  object 
 12  education_level            10127 non-null  object 
 13  income_category            10127 non-null  obj

### Escrita

In [14]:
df_dashboard.to_csv(
    "/home/tatiane/repositorios/data-projects/04_dashboards/customer_value_dashboard/data/customer_value_dashboard.csv",
    index=False
)